In [3]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

In [4]:
from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig

fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
    optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
)

accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"
base_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Re-init the tokenizer so it doesn't add padding or eos token
eval_tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    add_bos_token=True,
)

model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
from transformers import set_seed

set_seed(0)
def lm_gen(eval_prompt):
    model_input = eval_tokenizer(eval_prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = model.generate(**model_input, pad_token_id=eval_tokenizer.eos_token_id, max_new_tokens=128, do_sample=True)
        content = eval_tokenizer.decode(output[0], skip_special_tokens=True)
        # print(eval_tokenizer.decode(output[0], skip_special_tokens=True))
    # result_string  = content.replace(eval_prompt, "")
    return content.split("assistant\n")[1]
    # return content


def generate_prompt(content):
    system = "你是一個厲害的股市分析師，致力於幫助使用者的股市問題"
    content = content
    messages = [
        {
            "role": "system",
            "content": system,
        },
        {"role": "user", "content": content},
    ]
    tokenized_chat  = eval_tokenizer.apply_chat_template(messages, tokenize=False)
    # print(tokenized_chat)
    return tokenized_chat 

def lm_invoke(input):
    output = lm_gen(generate_prompt(input))
    return output

In [7]:
# content = ''' 請依據{業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢}對於鈦昇的影響，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關
# 在積體電路產業的領航帶動下，南科2022年營業額達1兆4,833.74億元，成長35.48％，未來，南科將持續持續布局前瞻產業，導入學研技術，厚植園區研發量能、打造兼具科技與環境永續的新世代園區，並擴散南科的成熟產業聚落效應，以強化區域經濟韌性。
# '''
# out = lm_invoke(content)
# print(out)

In [8]:
def lm_response(instruction, count):
    text = ''
    for i in range(count):
        try:
            text = lm_invoke(instruction)
            print(text)
        except:
            pass
    # print(text)

    if text.find('非常不好') > 0:
        score = -2
    elif text.find('非常好') > 0:
        score = 2
    elif text.find('不好') > 0:
        score = -1
    elif text.find('好') > 0:
        score = 1
    else:
        score = 0

    return score  # 分數

In [9]:
def calculate_return(signal, threshold):
    # signal = [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance = 10000000
    init_balance = balance

    for item in signal:
        score = float(item[1])
        price = float(item[2])

        if hold == 0:
            if score == 2:
                hold += int(balance/price)
                balance = 0
            elif score == 1:
                hold += int((balance/2)/price)
                balance = int(balance/2)
        elif hold > 0:
            if score < 0:
                balance += price*hold
                hold = 0
            elif score == 2:
                hold += int(balance/price)
                balance = 0
    
    balance += hold*price
    last_price = float(signal[len(signal) - 1][2])
    bnh = last_price / float(signal[0][2])
    r = balance/init_balance

    return round(r, 5), round(bnh, 5)

In [10]:
# def mutate_prompt(p_mutate):
#     prompt = '請用繁體中文回答以下提示的變體，同時保留語意：'
#     response = lm_invoke(prompt + p_mutate)
#     return response

# out = mutate_prompt('對業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢。')
# print(out)

In [11]:
# main cell
regenerate_count = 1
prompt_mutate_count = 100

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

# stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
# stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]
# stock_names = ["統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
stock_ids = ["8027"]
stock_names = ["鈦昇"]

p_mutate = '業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢'
rtn_ = 0

for i in range(prompt_mutate_count):
    # 變異
    # p_mutate_2 = "業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢"

    rtns = []
    bnhs = []
    raw_data = []
    for idx, stock_id in enumerate(stock_ids):
        print(f'變異{i + 1}次\t執行{stock_id} {stock_names[idx]}')
        general_prompt = f'請依據"{p_mutate}"對{stock_names[idx]}的影響，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關'
        folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
        files = os.listdir(folder_path)
        sorted_files = sorted(files)

        # 載入買進價格
        df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
        price_np = df.to_numpy()
        prices = price_np.tolist()  # [時間,價格]

        signal_2 = []

        for price in prices:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(price[0], '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_files):
                if file_name == price[0]:
                    with open(folder_path + file_name, encoding='utf-8') as f:
                        news = f.read()
                        # print(news)
                        prompt = general_prompt + "\n" + news
                        # gemini的回應，回傳帶有買入訊號的list
                        sig = lm_response(prompt, regenerate_count)
                        print(sig, prompt)
                    break
            
            # print(sig)
            if sig != 0 :
                signal_2.append([price[0], sig, price[1]])

        signal_2_np = np.array(signal_2)
        # 回傳 IRR 以及總額投入
        rtn, bnh = calculate_return(signal_2_np, 0)
        rtns.append(rtn)
        bnhs.append(bnh)
        raw_data.append(signal_2_np.tolist())
        # print("rtn", irr)
        # print("總額投入報酬率", bnh)

    # 輸出結果
    write_folder = "evaluate_output/output"
    count = 1
    while True:
        if os.path.exists(write_folder + str(count)):
            count += 1
            continue
        else:
            with open(write_folder + str(count), "a", encoding="UTF-8") as f:
                content = (f'提示：{prompt}\n'
                           f'開始日期 {start_time.strftime("%Y%m%d")},'
                           f'結束日期 {end_time.strftime("%Y%m%d")}\n')
                for j in range(len(stock_ids)):
                    content += (f'股票代號：{stock_ids[j]}\t'
                                f'依照分析報酬率:{rtns[j]}\t'
                                f'總額投入報酬率:{bnhs[j]}\n')
                content += f'變異{i + 1}次，平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{1.170022}\n)'
                content += f'原始數據{raw_data}'
                f.write(content)
            print("寫入" + write_folder + str(count))
            break

    # 判斷是否更改變異
    # if sum(rtns) / len(rtns) > rtn_:
    #     print("更改變異提示為:", p_mutate_2)
    #     p_mutate = p_mutate_2
    #     rtn_ = sum(rtns) / len(rtns)

# todo testing


變異1次	執行8027 鈦昇

Based on the information provided, I will analyze the impact of the company's convertible bond conversion on the stock price.

**First Conversion (Unsecured Convertible Bond)**

* The company converted 0 shares of convertible bond to common stock, with a face value of NT$0.
* The new shares have the same rights and obligations as the original issued shares.
* The converted shares will be directly credited to the account of the bondholders.
* The impact on the stock price is expected to be minimal, as the number of converted shares is zero.

Rating: #非常好

**Second Conversion (Secured Convert
2 請依據"業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢"對鈦昇的影響，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關
一、依據：發行人募集與發行有價證券處理準則第34條及本公司國內第三次無擔保轉換公司債發行及轉換辦法第10條、第13條及第15條規定辦理。 二、本公司國內第三次無擔保轉換公司債111年第4季經債權人提出申請轉換普通股，申請轉換公司債0張，合計轉換普通股0股，每股面額新台幣10元，合計新台幣0元。 三、新股權利義務：與原發行股份相同。 四、轉換後之新股上櫃：轉換後之普通股股票直接撥入債券持有人之集保帳戶。 五、特此公告。一、依據：發行人募集與發行有價證券處理準則第34條及本公司國內第二次有擔保轉換公司債發行及轉換辦法第10條、第13條及第15條規定辦理。 二

NameError: name 'p_mutate_2' is not defined

In [14]:
content = (
                           f'開始日期 {start_time.strftime("%Y%m%d")},'
                           f'結束日期 {end_time.strftime("%Y%m%d")}\n')
for j in range(len(stock_ids)):
                    content += (f'股票代號：{stock_ids[j]}\t'
                                f'依照分析報酬率:{rtns[j]}\t'
                                f'總額投入報酬率:{bnhs[j]}\n')
content += f'變異{i + 1}次，平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{1.170022}\n)'
content += f'原始數據{raw_data}'


print(content)



開始日期 20230101,結束日期 20231231
股票代號：8027	依照分析報酬率:1.41281	總額投入報酬率:1.336
變異1次，平均報酬率:1.41281，平均總額投入報酬率:1.170022
)原始數據[[['20230103', '2', '50.0'], ['20230303', '1', '52.3'], ['20230307', '2', '53.3'], ['20230313', '2', '51.9'], ['20230329', '1', '53.4'], ['20230410', '1', '56.6'], ['20230413', '2', '54.6'], ['20230414', '2', '53.4'], ['20230419', '2', '55.1'], ['20230505', '1', '50.3'], ['20230508', '2', '49.7'], ['20230515', '1', '43.8'], ['20230524', '2', '46.2'], ['20230601', '1', '45.3'], ['20230602', '2', '52.0'], ['20230605', '2', '52.6'], ['20230606', '1', '55.6'], ['20230612', '2', '59.9'], ['20230615', '1', '62.2'], ['20230616', '1', '59.9'], ['20230626', '-1', '60.7'], ['20230704', '2', '57.4'], ['20230713', '1', '57.8'], ['20230714', '2', '57.9'], ['20230718', '1', '63.4'], ['20230724', '2', '66.0'], ['20230731', '1', '63.8'], ['20230804', '2', '65.0'], ['20230807', '2', '71.5'], ['20230808', '1', '78.8'], ['20230810', '1', '69.9'], ['20230811', '1', '68.0'], ['20230817', '2', '72.

In [41]:
start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

# stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]
stock_ids = ["2912"]

# 變異
irrs = []
bnhs = []
raw_data = []
generate_prompt = "請參考以下資訊回答，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關"
for stock_id in stock_ids:
    print(f'執行{stock_id}')
    folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
    files = os.listdir(folder_path)
    sorted_files = sorted(files)

    # 載入買進價格
    df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
    price_np = df.to_numpy()
    prices = price_np.tolist()  # [時間,價格]

    signal_2 = []

    for price in prices:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(price[0], '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        sig = 0
        for index, file_name in enumerate(sorted_files):
            if file_name == price[0]:
                print(file_name)
                with open(folder_path + file_name, encoding='utf-8') as f:
                    news = f.read()
                    # lm的回應，回傳帶有買入訊號
                    sig = lm_response(generate_prompt(news))
                break

        signal_2.append([price[0], sig, price[1]])
        print(sig, "sd")


#     signal_2_np = np.array(signal_2)
#     # 回傳 IRR 以及總額投入
#     irr, bnh = calculate_irr(signal_2_np, 0)
#     irrs.append(irr)
#     bnhs.append(bnh)
#     raw_data.append(signal_2_np.tolist())
#     # print("每月內部報酬率", irr)
#     # print("總額投入報酬率", bnh)

# # 輸出結果
# write_folder = "evaluate_output/output"
# count = 1
# while True:
#     if os.path.exists(write_folder + str(count)):
#         count += 1
#         continue
#     else:
#         with open(write_folder + str(count), "a", encoding="UTF-8") as f:
#             content = (f'提示：{p_mutate_2}\n mutate from {p_mutate}\n'
#                         f'開始日期 {start_time.strftime("%Y%m%d")},'
#                         f'結束日期 {end_time.strftime("%Y%m%d")}\n')
#             for j in range(len(stock_ids)):
#                 content += (f'股票代號：{stock_ids[j]}\t'
#                             f'每月內部報酬率:{irrs[j]}\t'
#                             f'總額投入報酬率:{bnhs[j]}\n')
#             content += f'變異{i + 1}次，平均內部報酬率:{sum(irrs) / len(irrs)}，平均總額投入報酬率:{sum(bnhs) / len(bnhs)}\n)'
#             content += f'原始數據{raw_data}'
#             f.write(content)
#         print("寫入" + write_folder + str(count))
#         break


執行2912
20230103
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

請參考以下資訊回答，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關<|eot_id|><|start_header_id|>user<|end_header_id|>

2022年聖誕節假期因為博客來網路書店一位清潔阿姨「假承攬、真僱傭」事件，引起社會一陣軒然大波。素來形象良好，一直是網路書店領導者的博客來竟會發生與在公司打掃20幾年被要求打卡上下班的阿姨，簽定的是「清潔服務承攬合約書」，以致於阿姨不僅沒有勞健保、沒有三節禮金、沒有退休金，在阿姨的諮詢律師一篇臉書發文後，連法律學者都跳出來說道，無法接受一方面跟博客來下單買「21世紀資本論」，一方面看著博客來優雅地拿著刀叉吃弱勢者人肉。
撰文：溫正中通膨、升息導致全球消費需求疲弱，未來一年，台灣應該會強化發展內需產業，以彌補外銷不振的缺口。在內需產業中，應該還是以業績穩定的食品工業為首選，包括卜蜂（1215）、統一（1216）、聯華（1229）、聯華食（1231）、德麥（1264）、中華食（4205）都是不錯的食品類股，在股市動盪之際相形穩定，值得投資人注意。飼料肉品大廠卜蜂最近加大資本性支出，計劃在台南後壁區投資養雞場，不僅斥資 6,866 萬元，買下台南後壁區白沙屯約 2 千坪的土地，並花費 2 千萬元購買全新養殖設備，另外又投資 6.2 億元在桃園龜山工業區建立 560 坪工廠，預計 2024 年完工後將分布於桃園、汐止、中和、泰山各直營所的雞肉分切廠商全部遷入，集中管理。2021 年下半年開始，卜蜂遭遇「黃小玉」原料成本大漲，獲利受創，2021 年稅後純益年減 18.2%，每股獲利 5.06 元，配息 3 元、配股 1 元。2022 年上半年雞價、飼料、豬肉價格都上揚，獲利增加，預料全年度營收應可再創佳績。統一近年已經轉型為控股公司，集團為國內食品暨零售通路龍頭，屬於民生必需品，受景氣影響小，具防禦性。除了食品本業營運穩健，另由統一超主導流通事業，業績穩定成長。統一近年衝刺水市場，市占率約 25%，是國內最大水產品生產商，包括機能水、純水、礦泉水、氣泡水等 4 類，其中機能水產值約 32 億元，氣泡水產值 9 億元，礦泉水 6.5